In [1]:
# video Read

import cv2 
from ultralytics import YOLO  # truck detection ke liyea lib 

 model = YOLO("yolov8n.pt")            # YOLO() loads the trained model

# This creates a Python set containing five vehicle names.
vehicle_name = {
    "bicycle",
    "car",
    "motorcycle",
    "bus",
    "truck"
}   

# cv2.VideoCapture() opens the video file.
cap = cv2.VideoCapture(r"C:\Users\91936\Downloads\kaka.mp4")        

# set() isliye use kiya taaki duplicate count na ho 
counted_truck_ids = set()

# ye loop tab tak chalega jab tak file open hai 
# cap.isOpened() check karta hai ki video source available hai ya nahi
while cap.isOpened(): 
    ret, frame = cap.read() # cap.read() ek ek frame read karta hai
                            # ret is True when a frame is successfully read.  ((ret: True/False batata hai ki frame mila ya nahi. Agar video end ho gayi to ret=False))
                            # frame contains the image data. , (( frame: Actual image data (numpy array form mein))
    
    if not ret:             # agar video khatam ho gaya
        break
    
    # persist=True → har frame mein same object ko same ID milti hai (tracking)
    results = model.track(frame, persist=True, classes=[TRUCK_CLASS_ID], verbose=False)
    # classes=[7] → sirf trucks detect 
    
    if results[0].boxes is not None:
        boxes = results[0].boxes  # sab detected boxes
        
        for box in boxes:
            # Bounding box coordinates nikalo (x1, y1, x2, y2)
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # Confidence score (kitna confident hai model)
            conf = float(box.conf[0])

            # Tracking ID (har truck ka unique ID)
            track_id = int(box.id[0]) if box.id is not None else -1
            
            label = f"Truck ID:{track_id}  {conf:.2f}"          
            cv2.putText(frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

            if track_id != -1 and track_id not in counted_truck_ids:
                counted_truck_ids.add(track_id)  # naya truck mila → count mein add karo
                
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # (0, 255, 0) = green color, 2 = line thickness
            
    total = len(counted_truck_ids)  # total truck count
    cv2.putText(frame, f"Total Trucks: {total}", (20, 40),
    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)
    
    #gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) #Frame ko grayscale mein convert karta hai
    
    cv2.imshow('frame', frame) # Ek window kholta hai jisme frame show hota ha
    if cv2.waitKey(10) & 0xFF == ord('q'):  # 10 milisecond wait karega 
        break
    
cap.release()
cv2.destroyAllWindows()

print(f" Total Trucks Detected: {len(counted_truck_ids)}")

 Total Trucks Detected: 0
